In [33]:
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc)

import warnings
warnings.filterwarnings('ignore')
RANDOM_STATE = 100

## Data splitting

In [27]:
def split_data(path, delta=True):
    #Load file
    data = pd.read_csv(path)
    
    #Get features
    feature_names = [col for col in data.columns if col not in ['label', 'type', 'person_id', 'recording']]
    if not delta:
        feature_names = [col for col in feature_names if not col.startswith('d')]


    X = data[feature_names].values
    y = data['label'].values

    #Train/test split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y
    )

    print(f"Training: {X_train.shape[0]} samples")
    print(f"Test: {X_test.shape[0]} samples")
    print(f"Train distribution - Healthy: {sum(y_train==0)}, Pathological: {sum(y_train==1)}")

    #Feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return {
        "X_train": X_train,
        "X_test": X_test,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler
    }

In [50]:
phrase_data = split_data('data/preped/bin-phrase.csv', delta=True)

Training: 1014 samples
Test: 254 samples
Train distribution - Healthy: 507, Pathological: 507


## SVM training

In [37]:
def train_svm(data, kernel, c, sigma=None):

    if kernel == 'rbf':
        model = SVC(
            C=c,
            kernel='rbf',
            gamma=1 / (2 * (sigma ** 2)),
            random_state=RANDOM_STATE,
            probability=True,
            class_weight='balanced'
        )

    elif kernel == 'poly':
        model = SVC(
            C=c,
            kernel='poly',
            degree=2,
            random_state=RANDOM_STATE,
            probability=True,
            class_weight='balanced'
        )

    # Train on scaled data
    model.fit(data['X_train_scaled'], data['y_train'])

    # Training accuracy
    training_accuracy = model.score(
        data['X_train_scaled'],
        data['y_train']
    )

    # Predictions on test set
    y_pred = model.predict(data['X_test_scaled'])

    # Metrics
    accuracy = accuracy_score(data['y_test'], y_pred)
    precision = precision_score(data['y_test'], y_pred)
    recall = recall_score(data['y_test'], y_pred)
    f1 = f1_score(data['y_test'], y_pred)

    return (
        training_accuracy,
        accuracy,
        precision,
        recall,
        f1
    )

In [41]:
#rec_types = ['phrase', 'a', 'i', 'u']
kernel_types = ['rbf', 'poly']
c_values = [0.1, 1, 10, 100, 1000]
sigma_values = [0.05, 0.5, 1, 5, 10]
#sigma_values = [0.05, 0.5, 5]

In [42]:
results = []

# for rec_type in rec_types:
for c in c_values:
    for kernel in kernel_types:

        if kernel == 'rbf':
            for sigma in sigma_values:

                #name = f"{rec_type}-{kernel}-{c}-{sigma}"
                name = f"{kernel}-{c}-{sigma}"

                training_accuracy, accuracy, precision, recall, f1 = \
                    train_svm(phrase_data, kernel, c, sigma)

                results.append({
                    'model': name,
                    'training_accuracy': training_accuracy,
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1_score': f1
                })

        elif kernel == 'poly':

            #name = f"{rec_type}-{kernel}-{c}"
            name = f"{kernel}-{c}"
            
            training_accuracy, accuracy, precision, recall, f1 = train_svm(phrase_data, kernel, c)

            results.append({
                'model': name,
                'training_accuracy': training_accuracy,
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1_score': f1
            })

In [43]:
results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("RESULTS")
print("="*50)
print(results_df.to_string(index=False))


RESULTS
        model  training_accuracy  accuracy  precision   recall  f1_score
 rbf-0.1-0.05           1.000000  0.500000   0.500000 1.000000  0.666667
  rbf-0.1-0.5           1.000000  0.500000   0.500000 1.000000  0.666667
    rbf-0.1-1           1.000000  0.559055   0.532189 0.976378  0.688889
    rbf-0.1-5           0.747535  0.755906   0.815534 0.661417  0.730435
   rbf-0.1-10           0.722880  0.732283   0.855422 0.559055  0.676190
     poly-0.1           0.624260  0.625984   0.970588 0.259843  0.409938
   rbf-1-0.05           1.000000  0.500000   0.500000 1.000000  0.666667
    rbf-1-0.5           1.000000  0.503937   0.501976 1.000000  0.668421
      rbf-1-1           1.000000  0.688976   0.655844 0.795276  0.718861
      rbf-1-5           0.849112  0.767717   0.788136 0.732283  0.759184
     rbf-1-10           0.775148  0.759843   0.794643 0.700787  0.744770
       poly-1           0.795858  0.712598   0.787234 0.582677  0.669683
  rbf-10-0.05           1.000000  0.500000

## Random forest training

In [51]:
def train_rf(data, n_estimators=100, max_depth=None):
    
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=RANDOM_STATE,
        class_weight='balanced',
        n_jobs=-1
    )

    # Full training
    model.fit(data['X_train'], data['y_train'])

    training_accuracy = model.score(
        data['X_train'],
        data['y_train']
    )

    # Metrics on test set
    y_pred = model.predict(data['X_test'])

    accuracy = accuracy_score(data['y_test'], y_pred)
    precision = precision_score(data['y_test'], y_pred)
    recall = recall_score(data['y_test'], y_pred)
    f1 = f1_score(data['y_test'], y_pred)

    return training_accuracy, accuracy, precision, recall, f1

In [55]:
results = []

#for rec_type in rec_types:
for n_estimators in [10, 50, 100]:
    for max_depth in [None, 5, 10, 20]:

        #name = f"{rec_type}-RF-{n_estimators}-{max_depth}"
        name = f"RF-{n_estimators}-{max_depth}"

        training_accuracy, accuracy, precision, recall, f1 = train_rf(
            phrase_data,
            n_estimators=n_estimators,
            max_depth=max_depth
        )

        results.append({
            'model': name,
            'training_accuracy': training_accuracy,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        })


results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("RESULTS")
print("="*50)
print(results_df.to_string(index=False))


RESULTS
      model  training_accuracy  accuracy  precision   recall  f1_score
 RF-10-None           0.992110  0.740157   0.790476 0.653543  0.715517
    RF-10-5           0.847140  0.779528   0.825688 0.708661  0.762712
   RF-10-10           0.975345  0.759843   0.800000 0.692913  0.742616
   RF-10-20           0.992110  0.736220   0.783019 0.653543  0.712446
 RF-50-None           1.000000  0.771654   0.800000 0.724409  0.760331
    RF-50-5           0.874753  0.779528   0.838095 0.692913  0.758621
   RF-50-10           0.998028  0.759843   0.794643 0.700787  0.744770
   RF-50-20           1.000000  0.767717   0.798246 0.716535  0.755187
RF-100-None           1.000000  0.771654   0.805310 0.716535  0.758333
   RF-100-5           0.875740  0.791339   0.855769 0.700787  0.770563
  RF-100-10           0.992110  0.779528   0.814159 0.724409  0.766667
  RF-100-20           1.000000  0.767717   0.803571 0.708661  0.753138


## Logistic regression training

In [56]:
def train_lr(data):
    model = LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=1000)
    
    # Train on the full scaled training set
    model.fit(data["X_train_scaled"], data["y_train"])

    y_train_pred = model.predict(data["X_train_scaled"])
    training_accuracy = accuracy_score(data["y_train"], y_train_pred)
    
    # Metrics on scaled test set
    #if "X_test_scaled" in data and "y_test" in data:
    y_pred = model.predict(data['X_test_scaled'])

    accuracy = accuracy_score(data['y_test'], y_pred)
    precision = precision_score(data['y_test'], y_pred)
    recall = recall_score(data['y_test'], y_pred)
    f1 = f1_score(data['y_test'], y_pred)
    
    # print("--- Test Set Metrics ---")
    # print(f"Accuracy:  {accuracy:.4f}")
    # print(f"Precision: {precision:.4f}")
    # print(f"Recall:    {recall:.4f}")
    # print(f"F1 Score:  {f1:.4f}")
        
    return training_accuracy, accuracy, precision, recall, f1

In [58]:
results = []

training_accuracy, accuracy, precision, recall, f1 = train_lr(phrase_data)
results.append({
    'training_accuracy': training_accuracy,
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1
})

results_df = pd.DataFrame(results)
print("\n" + "="*50)
print("RESULTS")
print("="*50)
print(results_df.to_string(index=False))


RESULTS
 training_accuracy  accuracy  precision  recall  f1_score
          0.835306  0.767717   0.769841 0.76378  0.766798
